In [1]:
from EDTpy.ChipElements import *
import numpy as np
import copy
from EDTpy import settings
import EDTpy
from EDTpy import EmptyGeometry
from EDTpy import EmptyPath
from EDTpy.settings import *
import gdspy

# setting cpw parameters
CPW.default_values['S'] = 25
CPW.default_values['W'] = 16.3
CPW.default_values['layer'] = 0

dc_s = 1.5
dc_w = 1.5

In [2]:
class Marker(EmptyGeometry):
    default_values = {
        "a": 20,
        "b": 100,
        "layer": 0,
    }

    def _drawing(self, values):
        self.name = "Marker"
        
        a = values['a']
        b = values['b']
        layer = values['layer']
        
        self + gdspy.Rectangle([-b/2-b/10, -b/2-b/10],
                               [b/2+b/10, b/2+b/10], layer = layer)
        
        mark = gdspy.boolean(gdspy.Rectangle([-b/2, -a/2],[b/2, a/2]), gdspy.Rectangle([-a/2, -b/2],
                               [a/2, b/2]), 'or', layer = layer)
        self - mark
        self.add_port([0, 0], 0)
        
class Marker_set(EmptyGeometry):
    default_values = {
        "a": 200,
        "b": 200,
        "bm": 50,
        "layer": 0,
    }

    def _drawing(self, values):
        self.name = "Marker"
        
        a = values['a']
        b = values['b']
        bm = values['bm']
        am = bm/5
        layer = values['layer']
        
        self.add_port([0, 0], 0)
        self.add_port([-a/2, b/2], 0)
        self.add_port([a/2, b/2], 0)
        self.add_port([a/2, -b/2], 0)
        self.add_port([-a/2, -b/2], 0)
        
        m1 = Marker(a=am, b=bm)
        m1.merge_with(self.ports[1], 0)
        self+m1
        
        m1 = Marker(a=am, b=bm)
        m1.merge_with(self.ports[2], 0)
        self+m1
        
        m1 = Marker(a=am, b=bm)
        m1.merge_with(self.ports[3], 0)
        self+m1
        
        m1 = Marker(a=am, b=bm)
        m1.merge_with(self.ports[4], 0)
        self+m1
        

In [3]:
class CPW_C(EmptyGeometry):
    default_values = {
        "s": CPW.default_values['S'],
        "w": CPW.default_values['W'],
        "N": 2, #number of odd elements
        "l": 60, 
        "a": 3.3, # width of finger
        "b": 3.3, # gap between fingers
        "g": 15, # gap between fingers and GND
        "contact_pad_size": 200,
        "layer": 0,
    }

    def _drawing(self, values):
        self.name = "CPW_C"
        w = values['w']
        s = values['s']
        N = int((values['N'])/2)
        a = values['a']
        b = values['b']
        l = values['l']
        g = values['g']
        gnd = 30
        
        contact_pad_size = values['contact_pad_size']
        layer = values['layer']
        
        self + gdspy.Rectangle([0, -s/2-w],
                               [10, s/2+w], layer = 0)
        self - gdspy.Rectangle([0, -s/2],
                               [10, s/2], layer = 0)
        
        L = l+gnd
        jo = (N)*(a+b)*2+a/2
        jo_g = jo+g
        offset = L*0.4
        points = [(10, s/2+w), 
                  (offset, jo_g), (offset+L, jo_g), 
                  (offset+L, jo), (offset, jo), (10, s/2)]
        self + gdspy.Polygon(points)
        points = [(10, -s/2-w), 
                  (offset, -jo_g), (offset+L, -jo_g), 
                  (offset+L, -jo), (offset, -jo), (10, -s/2)]
        self + gdspy.Polygon(points)
        
        offset1 = offset*1.3
        self + gdspy.Rectangle([offset1, jo],
                               [offset1+l, -jo], layer = 0)
        
        self - gdspy.Rectangle([offset1,      a/2],
                               [offset1+l-b*3,  -a/2], layer = 0)
        
        for n in range(N):
            self - gdspy.Rectangle([offset1,     a/2+2*(a+b)*(n+1)-b],
                                   [offset1+l-b*3, a/2+2*(a+b)*(n+1)-b+a], layer = 0)
            self - gdspy.Rectangle([offset1,     -a/2-2*(a+b)*(n+1)+b],
                                   [offset1+l-b*3, -a/2-2*(a+b)*(n+1)+b-a], layer = 0)
            
            self - gdspy.Rectangle([offset1+b*3,     a/2+2*(a+b)*(n)+b],
                                   [offset1+l,     a/2+2*(a+b)*(n)+a+b], layer = 0)
            self - gdspy.Rectangle([offset1+b*3,    -a/2-2*(a+b)*(n)-b],
                                   [offset1+l,    -a/2-2*(a+b)*(n)-a-b], layer = 0)
        
        z = offset+L
        self + gdspy.Rectangle([z+100, -contact_pad_size/2-50],
                               [z+100+contact_pad_size+50, contact_pad_size/2+50], layer = 0)
        self - gdspy.Rectangle([z+100, -contact_pad_size/2],
                       [z+100+contact_pad_size, contact_pad_size/2], layer = 0)
        
        points = [(z, jo_g), 
                  (z+100, contact_pad_size/2+50), (z+100, contact_pad_size/2), 
                  (z, jo)]
        self + gdspy.Polygon(points)
        
        points = [(z, -jo_g), 
                  (z+100, -contact_pad_size/2-50), (z+100, -contact_pad_size/2), 
                  (z, -jo)]
        self + gdspy.Polygon(points)

        self.add_port([0, 0], 180)

In [4]:
class cap(EmptyGeometry):
    default_values = {
        "s": CPW.default_values['S'],
        "w": CPW.default_values['W'],
        "layer": 0,
    }

    def _drawing(self, values):
        self.name = "Marker"
        
        s = values['s']
        w = values['w']
        layer = values['layer']
        
        self + gdspy.Rectangle([0, -s/2-w],
                               [50, s/2+w], layer = layer)
        
        self.add_port([0, 0], 180)

In [10]:
class CPW_notch(EmptyGeometry):
    default_values = {
        "s": CPW.default_values['S'],
        "w": CPW.default_values['W'],
        "N": 2, #number of odd elements
        "l": 50, 
        "a": 3.3, # width of finger
        "b": 3.3, # gap between fingers
        "g": 15, # gap between fingers and GND
        "contact_pad_size": 300,
        "layer": 0,
    }

    def _drawing(self, values):
        self.name = "CPW_C"
        
        w = values['w']
        s = values['s']
        N = int((values['N'])/2)
        a = values['a']
        b = values['b']
        l = values['l']+b*2
        g = values['g']
        gnd = 30
        
        contact_pad_size = values['contact_pad_size']
        layer = values['layer']
        
        self + gdspy.Rectangle([0, -s/2-w],
                               [50, s/2+w], layer = 0)
        self - gdspy.Rectangle([0, -s/2],
                               [50, s/2], layer = 0)
            
        L = l+gnd
        jo = s/2
        jo_g = s/2+w

        z = 50
        self + gdspy.Rectangle([z+100, -contact_pad_size/2-50],
                               [z+100+contact_pad_size+50, contact_pad_size/2+50], layer = 0)
        self - gdspy.Rectangle([z+100, -contact_pad_size/2],
                       [z+100+contact_pad_size, contact_pad_size/2], layer = 0)
        
#         x1 = z
#         y1 = -s/2-w/2
        
#         x2 = z+100
#         y2 = -contact_pad_size/2-25
        
#         print(x1, y1, x2, y2)
        
#         lp = gdspy.RobustPath(
#             (x1, y1),
#             [w],
#             [x2],
#             ends=["extended"],
#             layer=[values['layer'], values['layer']],)
        
#         aa = 1
#         bb = 10
#         cc = 1 
#         dd = 10
#         ee = 1 
        
#         lp.segment(
#             (x2, y2),
#             width=[lambda u: aa+bb*u+cc*u**2+dd*u**3+ee*u**4 ],
#             offset=[
#                 lambda u: aa+bb*u+cc*u**2+dd*u**3+ee*u**4 , ],)

#         self + lp.to_polygonset()
        
        points = [(z, jo_g), 
                  (z+100, contact_pad_size/2+50), (z+100, contact_pad_size/2), 
                  (z, jo)]
        self + gdspy.Polygon(points)
        
        points = [(z, -jo_g), 
                  (z+100, -contact_pad_size/2-50), (z+100, -contact_pad_size/2), 
                  (z, -jo)]
        self + gdspy.Polygon(points)

        self.add_port([0, 0], 180)

In [5]:
class Notch2(EmptyGeometry):
    default_values = {
        "coupling_x": 900,
        "coupling_y": 10,
        "layer": 0,
        "r":150,
        "l": 270,
        "reflect": False,
    }

    def _drawing(self, values):
        
        coupling_x = values['coupling_x']
        coupling_y = values['coupling_y']
        layer = values['layer']
        l=values['l']
        r=values['r']
        reflect=values['reflect']
        
        res_path = [[-coupling_x/2-800, 1800], 
                    [-coupling_x/2-800, 1400], 
                    [-coupling_x/2, 1400], 
                    [-coupling_x/2, 1000], 
                    [-coupling_x/2-800, 1000],
                    [-coupling_x/2-800, 600],
                    [-coupling_x/2, 600],
                    [-coupling_x/2, (CPW.default_values['S']/2+CPW.default_values['W'])*2+coupling_y], 
                    [coupling_x/2, (CPW.default_values['S']/2+CPW.default_values['W'])*2+coupling_y],
                    [coupling_x/2, 1300] ]
        
        if reflect: 
            for i in range(len(res_path)):
                res_path[i][0] = -res_path[i][0]

        
        res_R =r
        res = CPW(res_path, res_R, layer=layer)
        print(res.length)
        self + res

        self.add_port([0, 0], 0)
        self.add_port(res.ports[1].position, -90)
        self.add_port(res.ports[0].position, 180)
        


In [6]:
class Marker_f(EmptyGeometry):
    default_values = {
        "a": 20,
        "b": 20,
        "layer": 0,
    }

    def _drawing(self, values):
        self.name = "Marker"
        
        a = values['a']
        b = values['b']
        layer = values['layer']
        
        self + gdspy.Rectangle([-a, b],
                               [0, 0], layer = layer)
        self + gdspy.Rectangle([a, -b],
                               [0, 0], layer = layer)
        
        for i in range(7):
            self + gdspy.Rectangle([a, 0.8*(i+1)+1.2*i],
                                   [0, 0.8*(i+1)+1.2*(i+1)], layer = layer)
        
        for i in range(7):
            self + gdspy.Rectangle([-a, -0.8*(i+1)-1.2*i],
                                   [0, -0.8*(i+1)-1.2*(i+1)], layer = layer)
                
        self.add_port([0, 0], 0)
        self.add_port([0, 0], 90)

In [7]:
sketch = EDTpy.EmptySketch()
test = Marker_f()
sketch.add_geometry(test)
test.show()

In [11]:
sketch = EDTpy.EmptySketch()
sketch + gdspy.Rectangle([-7000/2, -7000/2],
                         [7000/2, 7000/2], layer = 0)

sketch - gdspy.Rectangle([-7000/2 + 250, -7000/2 + 250],
                         [7000/2 - 250, 7000/2 - 250], layer = 0)

tx = 7000/2
ty = 7000/2
mtx = tx-5000
mty = ty-5000

sketch + gdspy.Rectangle([tx, ty],
                         [tx-5000, ty-5000], layer = 1)

sketch - gdspy.Rectangle([tx-250, ty-250],
                         [tx-5000 + 250, ty-5000+250], layer = 1)

sketch.add_port([-3000, 3000], -180) #Marker port  --  0
marker = Marker()
marker.merge_with(sketch.ports[0], 0)
sketch.add_geometry(marker)
sketch.circular_array(marker)

sketch.add_port([tx-500, ty-500], -180) #Marker port  --  1
marker = Marker(layer=1)
marker.merge_with(sketch.ports[1], 0)
sketch.add_geometry(marker)


sketch.add_port([tx-500, mty+500], -180) #Marker port  --  2
marker = Marker(layer=1)
marker.merge_with(sketch.ports[2], 0)
sketch.add_geometry(marker)
marker = Marker(layer=0)
marker.merge_with(sketch.ports[2], 0)
sketch.add_geometry(marker)


sketch.add_port([mtx+500, mty+500], -180) #Marker port  --  3
marker = Marker(layer=1)
marker.merge_with(sketch.ports[3], 0)
sketch.add_geometry(marker)

marker = Marker(layer=0)
marker.merge_with(sketch.ports[3], 0)
sketch.add_geometry(marker)


sketch.add_port([mtx+500, ty-500], -180) #Marker port  --  4
marker = Marker(layer=1)
marker.merge_with(sketch.ports[4], 0)
sketch.add_geometry(marker)

marker = Marker(layer=0)
marker.merge_with(sketch.ports[4], 0)
sketch.add_geometry(marker)

sketch.add_port([-2600, 2700], 180) #Marker port  --  5

res_path = [[50, 1200], [2000, 1200]]
res_R = 200
res = CPW(res_path, res_R)
CPW_c_r = CPW_C()
CPW_c_r.merge_with(sketch.ports[5], 0)
sketch.add_geometry(CPW_c_r)
res.merge_with(CPW_c_r.ports[0], 0)
sketch.add_geometry(res)

res_path = [[0, 0], [30, 0]]
res_u = CPW(res_path, res_R)
res_u.merge_with(res.ports[1], 0)
sketch.add_geometry(res_u)

c = cap(layer=0)
c.merge_with(res_u.ports[1], 0)
sketch.add_geometry(c)

c = cap(layer=1)
c.merge_with(res_u.ports[0], 0)
sketch.add_geometry(c)


res_path = [[50, 1200], [2000, 1200]]
res_top = CPW(res_path, res_R, layer=1)
res_top.merge_with(res.ports[1], 0)
sketch.add_geometry(res_top)
print(res.length+res_top.length)

sketch.add_port([-2600, 2200], 180) #Marker port  --  6

res_path = [[0, 1200], [3900, 1200]]
res_R = 200
res = CPW(res_path, res_R)
CPW_c_r = CPW_C()
CPW_c_r.merge_with(sketch.ports[6], 0)
sketch.add_geometry(CPW_c_r)
res.merge_with(CPW_c_r.ports[0], 0)
sketch.add_geometry(res)
print(res.length)

x0 = -2400
y0 = 2300
text = gdspy.Text("R7.4", 250, (x0, y0-20))
sketch + text 





sketch.add_port([-2600, 1700], 180) #Marker port  --  7

res_path = [[50, 1200], [2000, 1200]]
res_R = 200
res = CPW(res_path, res_R)
CPW_c_r = CPW_C()
CPW_c_r.merge_with(sketch.ports[7], 0)
sketch.add_geometry(CPW_c_r)
res.merge_with(CPW_c_r.ports[0], 0)
sketch.add_geometry(res)

res_path = [[0, 0], [30, 0]]
res_u = CPW(res_path, res_R)
res_u.merge_with(res.ports[1], 0)
sketch.add_geometry(res_u)

c = cap(layer=1)
c.merge_with(res_u.ports[0], 0)
sketch.add_geometry(c)

res_path = [[50, 1200], [2000, 1200]]
res_top = CPW(res_path, res_R, layer=1)
res_top.merge_with(res.ports[1], 0)
sketch.add_geometry(res_top)

c = cap(layer=1)
c.merge_with(res_top.ports[1], 0)
sketch.add_geometry(c)

ct = cap(layer=1)
ct.merge_with(res_top.ports[1], 0)
sketch.add_geometry(ct)

c = cap(layer=0)
c.merge_with(res_u.ports[1], 0)
sketch.add_geometry(c)

res_path = [[0, 0], [30, 0]]
res_b2 = CPW(res_path, res_R, layer=0)
res_b2.merge_with(ct.ports[0], 1)
sketch.add_geometry(res_b2)

c = cap(layer=0)
c.merge_with(res_b2.ports[0], 0)
sketch.add_geometry(c)


res_path = [[0, 0], [1200, 0]]
res_b3 = CPW(res_path, res_R, layer=0)
res_b3.merge_with(res_b2.ports[1], 0)
sketch.add_geometry(res_b3)

print(res.length+res_top.length)

sketch.add_port([-2600, 1200], 180) #Marker port  --  8

res_path = [[0, 1200], [5100, 1200]]
res_R = 200
res = CPW(res_path, res_R)
CPW_c_r = CPW_C()
CPW_c_r.merge_with(sketch.ports[8], 0)
sketch.add_geometry(CPW_c_r)
res.merge_with(CPW_c_r.ports[0], 0)
sketch.add_geometry(res)
print(res.length)



sketch.add_port([-2500, -700], 180) #Marker port  --  9
sketch.add_port([2300, -2500], -90) #Marker port  --  10

CPW_1 = CPW_notch(s=CPW.default_values['S'], w=CPW.default_values['W'])
CPW_1.merge_with(sketch.ports[9], 0)
sketch.add_geometry(CPW_1)

CPW_2 = CPW_notch(s=CPW.default_values['S'], w=CPW.default_values['W'])
CPW_2.merge_with(sketch.ports[10], 0)
sketch.add_geometry(CPW_2)

h = 700
res_path = [sketch.ports[9].position,
            [sketch.ports[10].position[0], sketch.ports[9].position[1]],
            [sketch.ports[10].position[0], sketch.ports[9].position[1]],
            sketch.ports[10].position]
res_R = 150
res = CPW(res_path, res_R)
sketch.add_geometry(res)

x0 = -2400
y0 = 1300
text = gdspy.Text("R5.68", 250, (x0, y0-20))
sketch + text 


sketch.add_port([-900, -700], 0) #Marker port  --  11
n2 = Notch2(coupling_y=CPW.default_values['S'])
n2.merge_with(sketch.ports[11], 0)
sketch.add_geometry(n2)
c = cap(layer=0)
c.merge_with(n2.ports[1], 0)
sketch.add_geometry(c)

x0 = -2400
y0 = -500
text = gdspy.Text("N5.124", 250, (x0, y0-20))
sketch + text 


sketch.add_port([900, -700], 180) #Marker port  --  12
n2 = Notch2(coupling_y=-CPW.default_values['W']*2-CPW.default_values['S'], layer=1)
n2.merge_with(sketch.ports[12], 0)
sketch.add_geometry(n2)
c = cap(layer=1)
c.merge_with(n2.ports[1], 0)
sketch.add_geometry(c)


# MARKERS FOR FLIP-CHIP
# tx = 7000/2
# ty = 7000/2
# mtx = tx-5000
# mty = ty-5000

sketch.add_port([tx-600, ty-600], 180) #Marker port  --  13
n2 = Marker_f(a=40,b=40)
n2.merge_with(sketch.ports[13], 0)
sketch.add_geometry(n2)
n2 = Marker_f(a=40,b=40,layer=1)
n2.merge_with(sketch.ports[13], 1)
sketch.add_geometry(n2)

sketch.add_port([mtx+600, ty-600], 180) #Marker port  --  14
n2 = Marker_f(a=40,b=40)
n2.merge_with(sketch.ports[14], 0)
sketch.add_geometry(n2)
n2 = Marker_f(a=40,b=40,layer=1)
n2.merge_with(sketch.ports[14], 1)
sketch.add_geometry(n2)

sketch.add_port([tx-600, mty+600], 180) #Marker port  --  15
n2 = Marker_f(a=40,b=40)
n2.merge_with(sketch.ports[15], 0)
sketch.add_geometry(n2)
n2 = Marker_f(a=40,b=40,layer=1)
n2.merge_with(sketch.ports[15], 1)
sketch.add_geometry(n2)

sketch.add_port([tx-600, (mty+ty)/2], 180) #Marker port  --  16
n2 = Marker_f(a=40,b=40)
n2.merge_with(sketch.ports[16], 0)
sketch.add_geometry(n2)
n2 = Marker_f(a=40,b=40,layer=1)
n2.merge_with(sketch.ports[16], 1)
sketch.add_geometry(n2)

sketch.add_port([mtx+600, (mty+ty)/2], 180) #Marker port  --  17
n2 = Marker_f(a=40,b=40)
n2.merge_with(sketch.ports[17], 0)
sketch.add_geometry(n2)
n2 = Marker_f(a=40,b=40,layer=1)
n2.merge_with(sketch.ports[17], 1)
sketch.add_geometry(n2)


sketch.add_port([(tx+mtx)/2, ty-600], 180) #Marker port  --  18
n2 = Marker_f(a=40,b=40)
n2.merge_with(sketch.ports[18], 0)
sketch.add_geometry(n2)
n2 = Marker_f(a=40,b=40,layer=1)
n2.merge_with(sketch.ports[18], 1)
sketch.add_geometry(n2)

sketch.add_port([(tx+mtx)/2, mty+600], 180) #Marker port  --  19
n2 = Marker_f(a=40,b=40)
n2.merge_with(sketch.ports[19], 0)
sketch.add_geometry(n2)
n2 = Marker_f(a=40,b=40,layer=1)
n2.merge_with(sketch.ports[19], 1)
sketch.add_geometry(n2)

sketch.add_port([(tx+mtx)/2, (mty+ty)/2], 180) #Marker port  --  20
n2 = Marker_f(a=40,b=40)
n2.merge_with(sketch.ports[20], 0)
sketch.add_geometry(n2)
n2 = Marker_f(a=40,b=40,layer=1)
n2.merge_with(sketch.ports[20], 1)
sketch.add_geometry(n2)

sketch.add_port([tx-670, ty-670], 180) #Marker port  --  21
n2 = Marker_f(a=30,b=30)
n2.merge_with(sketch.ports[21], 0)
sketch.add_geometry(n2)
n2 = Marker_f(a=30,b=30,layer=1)
n2.merge_with(sketch.ports[21], 1)
sketch.add_geometry(n2)

sketch.add_port([tx-670, mty+670], 180) #Marker port  --  22
n2 = Marker_f(a=30,b=30)
n2.merge_with(sketch.ports[22], 0)
sketch.add_geometry(n2)
n2 = Marker_f(a=30,b=30,layer=1)
n2.merge_with(sketch.ports[22], 1)
sketch.add_geometry(n2)

sketch.add_port([mtx+670, ty-670], 180) #Marker port  --  23
n2 = Marker_f(a=30,b=30)
n2.merge_with(sketch.ports[23], 0)
sketch.add_geometry(n2)
n2 = Marker_f(a=30,b=30,layer=1)
n2.merge_with(sketch.ports[23], 1)
sketch.add_geometry(n2)


sketch.add_port([tx-720, ty-720], 180) #Marker port  --  24
n2 = Marker_f(a=20,b=20)
n2.merge_with(sketch.ports[24], 0)
sketch.add_geometry(n2)
n2 = Marker_f(a=20,b=20,layer=1)
n2.merge_with(sketch.ports[24], 1)
sketch.add_geometry(n2)

sketch.assemble()
sketch.show()

3900.0
3900.0
3900.0
5100.0
5719.755592153877
5884.955592153877


Assembling Empty Sketch: 100%|████████████████████████████████████████████████████████| 63/63 [00:00<00:00, 299.48it/s]
